In [1]:
!pip install geopandas pyogrio



### Cell 1: Imports and config

In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path

# Paths to your existing cleaned files
DC_PATH = "data_centers_cleaned.csv"
PLANTS_PATH = "plants_cleaned.csv"
PRICES_PATH = "electricity_price_cleaned.csv"

# Output
OUTPUT_PATH = "county_features.csv"

### Cell 2: Load Census county shapefile 

In [3]:
# Census 2023 county boundaries, low-res 20m version (smaller download, plenty for analysis)
COUNTIES_URL = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_county_20m.zip"

counties = gpd.read_file(COUNTIES_URL)

# Keep only continental US (drop AK=02, HI=15, PR=72, and other territories)
EXCLUDE_STATEFP = {"02", "15", "60", "66", "69", "72", "78"}
counties = counties[~counties["STATEFP"].isin(EXCLUDE_STATEFP)].copy()

# Standardize FIPS as zero-padded 5-char string
counties["GEOID"] = counties["GEOID"].astype(str).str.zfill(5)

# Reproject to a US-friendly equal-area CRS for accurate distance calculations later
# (EPSG:5070 is NAD83 / Conus Albers, in meters)
counties = counties.to_crs("EPSG:5070")

print(f"Loaded {len(counties)} continental US counties")
counties[["GEOID", "NAME", "STATEFP", "ALAND"]].head()

Loaded 3109 continental US counties


,GEOID,NAME,STATEFP,ALAND
0,13027,Brooks,13,1277341276
1,31095,Jefferson,31,1476765697
2,51683,Manassas,51,25493247
3,56021,Laramie,56,6956355036
4,13135,Gwinnett,13,1116606767


### Cell 3: Build a state FIPS → state name lookup 

In [4]:
# Census state FIPS codes for the 48 continental + DC
STATE_FIPS_TO_NAME = {
    "01": "Alabama", "04": "Arizona", "05": "Arkansas", "06": "California",
    "08": "Colorado", "09": "Connecticut", "10": "Delaware", "11": "District of Columbia",
    "12": "Florida", "13": "Georgia", "16": "Idaho", "17": "Illinois",
    "18": "Indiana", "19": "Iowa", "20": "Kansas", "21": "Kentucky",
    "22": "Louisiana", "23": "Maine", "24": "Maryland", "25": "Massachusetts",
    "26": "Michigan", "27": "Minnesota", "28": "Mississippi", "29": "Missouri",
    "30": "Montana", "31": "Nebraska", "32": "Nevada", "33": "New Hampshire",
    "34": "New Jersey", "35": "New Mexico", "36": "New York", "37": "North Carolina",
    "38": "North Dakota", "39": "Ohio", "40": "Oklahoma", "41": "Oregon",
    "42": "Pennsylvania", "44": "Rhode Island", "45": "South Carolina",
    "46": "South Dakota", "47": "Tennessee", "48": "Texas", "49": "Utah",
    "50": "Vermont", "51": "Virginia", "53": "Washington", "54": "West Virginia",
    "55": "Wisconsin", "56": "Wyoming"
}

counties["state_name"] = counties["STATEFP"].map(STATE_FIPS_TO_NAME)
print(f"States represented: {counties['state_name'].nunique()}")

States represented: 49


### Cell 4: Spatial join data centers → counties


In [5]:
dc = pd.read_csv(DC_PATH)
print(f"Loaded {len(dc)} data centers. Columns: {list(dc.columns)}")

# Build GeoDataFrame from lat/lon
dc_gdf = gpd.GeoDataFrame(
    dc,
    geometry=gpd.points_from_xy(dc["lon"], dc["lat"]),
    crs="EPSG:4326"
).to_crs("EPSG:5070")  # match counties' CRS

# Spatial join: which county does each data center fall in?
dc_joined = gpd.sjoin(dc_gdf, counties[["GEOID", "geometry"]], how="left", predicate="within")

# Check unmatched (likely 0 or just a few)
unmatched = dc_joined["GEOID"].isna().sum()
print(f"Unmatched data centers: {unmatched} ({unmatched/len(dc_joined)*100:.1f}%)")

# Aggregate per county: count and total sqft
dc_per_county = dc_joined.groupby("GEOID").agg(
    dc_count=("id", "count"),
    dc_total_sqft=("sqft", "sum")  # sum ignores NaN by default
).reset_index()

print(f"Counties with at least one data center: {len(dc_per_county)}")
dc_per_county.head()

Loaded 1472 data centers. Columns: ['id', 'state', 'state_abb', 'county', 'county_id', 'lat', 'lon', 'type', 'sqft']
Unmatched data centers: 1 (0.1%)
Counties with at least one data center: 247


,GEOID,dc_count,dc_total_sqft
0,01069,1,10736.0
1,01071,3,7869085.0
2,01073,1,665569.0
3,01089,6,16255032.0
4,04003,1,7539.0


### Cell 5: Spatial join renewable plants → counties


In [6]:
plants = pd.read_csv(PLANTS_PATH)
print(f"Loaded {len(plants)} plants. Columns: {list(plants.columns)}")

plants_gdf = gpd.GeoDataFrame(
    plants,
    geometry=gpd.points_from_xy(plants["Longitude"], plants["Latitude"]),
    crs="EPSG:4326"
).to_crs("EPSG:5070")

plants_joined = gpd.sjoin(plants_gdf, counties[["GEOID", "geometry"]], how="left", predicate="within")
unmatched_p = plants_joined["GEOID"].isna().sum()
print(f"Unmatched plants: {unmatched_p}")

# Aggregate: total in-county renewable capacity
renew_per_county = plants_joined.groupby("GEOID").agg(
    plant_count=("Plant Code", "count"),
    renewable_mw_in_county=("Nameplate Capacity (MW)", "sum")
).reset_index()

renew_per_county.head()

Loaded 9294 plants. Columns: ['Plant Code', 'Plant Name', 'State', 'Latitude', 'Longitude', 'Energy Source', 'Nameplate Capacity (MW)']
Unmatched plants: 101


,GEOID,plant_count,renewable_mw_in_county
0,01015,1,7.4
1,01017,1,79.2
2,01019,1,29.2
3,01021,1,29.5
4,01033,1,227.0


### Cell 6: Renewable capacity within 50km of county centroid 

In [7]:
from scipy.spatial import cKDTree

# County centroids in meters (EPSG:5070)
county_centroids = counties.copy()
county_centroids["centroid"] = county_centroids.geometry.centroid
centroid_coords = np.array([(p.x, p.y) for p in county_centroids["centroid"]])

# Plant coordinates in meters
plant_coords = np.array([(p.x, p.y) for p in plants_gdf.geometry])
plant_capacity = plants_gdf["Nameplate Capacity (MW)"].fillna(0).values

# Build KDTree on plants and query 50km neighbors per county centroid
tree = cKDTree(plant_coords)
RADIUS_M = 50_000  # 50 km in meters

renewable_within_50km = []
for i, c in enumerate(centroid_coords):
    idx = tree.query_ball_point(c, r=RADIUS_M)
    renewable_within_50km.append(plant_capacity[idx].sum())

county_centroids["renewable_mw_within_50km"] = renewable_within_50km

# Slim table for merging
proximity_df = county_centroids[["GEOID", "renewable_mw_within_50km"]]
proximity_df.head()

,GEOID,renewable_mw_within_50km
0,13027,379.7
1,31095,818.2
2,51683,124.8
3,56021,503.6
4,13135,108.5


### Cell 7: Add electricity price (state-level → broadcast to counties)


In [8]:
prices = pd.read_csv(PRICES_PATH)
print(f"Prices: {list(prices.columns)}")

# prices has columns: state, electricity_price
# Join via state name
prices_to_merge = prices.rename(columns={"state": "state_name"})
counties_with_price = counties.merge(prices_to_merge, on="state_name", how="left")

missing_price = counties_with_price["electricity_price"].isna().sum()
print(f"Counties missing price: {missing_price}")

Prices: ['state', 'electricity_price']
Counties missing price: 1


### Cell 8: Assemble the master county feature table


In [9]:
# Start from counties (skeleton), left-join everything
features = counties_with_price[["GEOID", "NAME", "STATEFP", "state_name", "ALAND"]].copy()
features = features.rename(columns={"NAME": "county_name", "ALAND": "land_area_m2"})

# Merge electricity price back (already on counties_with_price)
features = features.merge(
    counties_with_price[["GEOID", "electricity_price"]], on="GEOID", how="left"
)

# Data center variables (fill NaN with 0 — no data center = 0, not missing)
features = features.merge(dc_per_county, on="GEOID", how="left")
features["dc_count"] = features["dc_count"].fillna(0).astype(int)
features["dc_total_sqft"] = features["dc_total_sqft"].fillna(0)
features["has_dc"] = (features["dc_count"] > 0).astype(int)  # binary target for ML

# Renewable in-county
features = features.merge(renew_per_county, on="GEOID", how="left")
features["plant_count"] = features["plant_count"].fillna(0).astype(int)
features["renewable_mw_in_county"] = features["renewable_mw_in_county"].fillna(0)

# Renewable within 50km
features = features.merge(proximity_df, on="GEOID", how="left")
features["renewable_mw_within_50km"] = features["renewable_mw_within_50km"].fillna(0)

# Land area in km², population density placeholder (filled in next cells)
features["land_area_km2"] = features["land_area_m2"] / 1e6

print(features.shape)
features.head()

(3109, 13)


,GEOID,county_name,STATEFP,state_name,land_area_m2,electricity_price,dc_count,dc_total_sqft,has_dc,plant_count,renewable_mw_in_county,renewable_mw_within_50km,land_area_km2
0,13027,Brooks,13,Georgia,1277341276,7.53,0,0.0,0,2,300.0,379.7,1277.341276
1,31095,Jefferson,31,Nebraska,1476765697,7.35,0,0.0,0,1,74.8,818.2,1476.765697
2,51683,Manassas,51,Virginia,25493247,9.35,5,881180.0,1,0,0.0,124.8,25.493247
3,56021,Laramie,56,Wyoming,6956355036,8.30,18,21738866.0,1,6,503.6,503.6,6956.355036
4,13135,Gwinnett,13,Georgia,1116606767,7.53,4,2155213.0,1,0,0.0,108.5,1116.606767


In [10]:
features.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(features)} county rows to {OUTPUT_PATH}")
print(f"Counties with data centers: {features['has_dc'].sum()}")
print(f"Mean dc_count: {features['dc_count'].mean():.2f}")

Saved 3109 county rows to county_features.csv
Counties with data centers: 247
Mean dc_count: 0.47
